# Global Description Generator Test

This notebook tests `GlobalDescriptionGenerator` with populated entity types, random entity and chunk samples, concurrent LLM calls, versioned `global_config` persistence, verbose prompts, and forced regeneration through debug mode.

In [1]:
import importlib
import json

from dotenv import load_dotenv

import dotvectordb.graph.generate_global_desc as global_desc_module
from dotvectordb.DotVDB import DotVectorDB
from dotvectordb.plugins.embedding_providers.openai_embeddings import OpenAIEmbeddingModel
from dotvectordb.plugins.llm_providers.openai_llms import OpenAiAgent

load_dotenv()
DATABASE_PATH = "test_db9.db"
SAMPLE_SIZE = 10

In [2]:
language_model = OpenAiAgent()
embedding_model = OpenAIEmbeddingModel()
db = DotVectorDB(
    path=DATABASE_PATH,
    DIM=1536,
    embedding_model=embedding_model,
    language_model=language_model,
    verbose=True,
)
await db.open()

Database connection opened and vec1 extension loaded successfully.
Default schema initialized successfully.
DotVectorDB initialized successfully.


In [3]:
entity_types = await db.get_entity_types(only_populated=True)
sources = await db.get_sources()
global_config = await db.get_global_config()

print("Populated entity types:")
print(json.dumps(entity_types, indent=2))
print("\nSources:")
print(json.dumps(sources, indent=2))
print("\nGlobal config before generation:")
print(json.dumps(global_config, indent=2))

Populated entity types:
{
  "person": "A named fictional or non-fictional individual human",
  "organization": "A group or institution of fictional or non-fictional people.",
  "location": "A fictional or non-fictional named place or geographic area, such as a city, country, or landmark",
  "substance": "A named edible substance, drug, element, or chemical compound",
  "creative_work": "A named non-academic focused artistic or literary creation, such as a book, movie, album, game, song, article, or painting",
  "concept": "A named idea, theory, framework, or principle, such as relativity, democracy, RAG, or the butterfly effect"
}

Sources:
[
  {
    "id": "-8917888661152971970",
    "name": "tests/shrek.txt"
  }
]

Global config before generation:
{
  "current_version_count": 88,
  "description_version": 0,
  "description": null
}


In [4]:
global_desc_module = importlib.reload(global_desc_module)
GlobalDescriptionGenerator = global_desc_module.GlobalDescriptionGenerator

generator = GlobalDescriptionGenerator(
    conn=db.conn,
    lock=db.lock,
    language_model=language_model.generate,
    entity_types=entity_types,
    sources=sources,
    global_config=global_config,
    save_global_description_func=db.update_global_description,
    verbose=True,
    debug_mode=True,
)

result = await generator.generate(sample_size=SAMPLE_SIZE)
result

Generating a global description because debug mode is enabled.
Sending 6 entity-type description request(s) and 1 chunk-description request concurrently.
Sampled 10 person entity/entities.
Prompt sent to the LLM for entity type 'person':

<Role>
You are a Knowledge Graph Specialist responsible for reading a list of randomly sampled entities of a specific type with the objective
of trying to figure out a global description of what the full corpus could be about.
</Role>

<Instructions>
   - Use the source names in the '<Source_Names>' section as context for identifying what the graph contains.
   - Carefully read the '<Type>' section to understand the type of entities that were sampled from the graph database.
   - Read the '<Sample>' section to understand the sampled entities of the specific type.
   - Based on the information in the '<Type>' and '<Sample>' sections, observe the commonalities and patterns among the sampled entities to infer a global description of what the full corpus 

{'global_description': 'The corpus primarily centers around the animated film "Shrek" and its associated themes, characters, and settings. It features a diverse array of fictional characters, including the gruff yet kind-hearted ogre Shrek, the cursed Princess Fiona, and the villainous Lord Farquaad, each embodying distinct roles within a narrative that blends humor, adventure, and romance. The exploration of fictional organizations, such as the Merry Men, and locations like DuLoc and a dragon-guarded castle, further enriches the narrative landscape, highlighting the contrast between idealized settings and the challenges faced by the characters. Additionally, the corpus delves into whimsical culinary elements, showcasing humorous and unconventional food items that reflect the playful tone of the franchise. Central themes include love, identity, transformation, and the subversion of traditional fairy tale tropes, emphasizing that true beauty and heroism come from within. Overall, the co

In [5]:
updated_global_config = await db.get_global_config()

print("Current version:", updated_global_config["current_version_count"])
print("Description version:", updated_global_config["description_version"])
print("\nSaved description:")
print(json.dumps(json.loads(updated_global_config["description"]), indent=2))

Current version: 88
Description version: 88

Saved description:
{
  "global_description": "The corpus primarily centers around the animated film \"Shrek\" and its associated themes, characters, and settings. It features a diverse array of fictional characters, including the gruff yet kind-hearted ogre Shrek, the cursed Princess Fiona, and the villainous Lord Farquaad, each embodying distinct roles within a narrative that blends humor, adventure, and romance. The exploration of fictional organizations, such as the Merry Men, and locations like DuLoc and a dragon-guarded castle, further enriches the narrative landscape, highlighting the contrast between idealized settings and the challenges faced by the characters. Additionally, the corpus delves into whimsical culinary elements, showcasing humorous and unconventional food items that reflect the playful tone of the franchise. Central themes include love, identity, transformation, and the subversion of traditional fairy tale tropes, empha

In [ ]:
await db.close()